In [2]:
import matplotlib.pyplot as plt
import pickle
import numpy as np
import joblib as jl
from tqdm import tqdm

from functions_synthetic_data_generation import *
from functions_synthetic_data_analysis import *
from functions_hmm import *


plt.style.use('../paper.mplstyle')

print(plt.get_backend())
%matplotlib qt5
print(plt.get_backend())

module://matplotlib_inline.backend_inline


qt5agg


# Simulations generations

## Parameters

In [46]:
p_cw_reward = 0.8
p_ccw_reward = 0
p_cw_init = 0.5
steps_number = 40
noise_amplitude = 0.1
drift_matrix = np.array([[ 0.05 , -0.05],
                         [ 0    , 0   ]])
drift_init = 0.0

args_dict = {'p_cw_reward': p_cw_reward, 'p_ccw_reward': p_ccw_reward, 'p_cw_init': p_cw_init, 'steps_number': steps_number, 'noise_amplitude': noise_amplitude, 'drift_matrix': drift_matrix, 'drift_init': drift_init}

number_of_simulations_perset = 50

n_simulations_list = [number_of_simulations_perset]*100

start_index = 0

# fit_type = 'aic'
fit_type = 'score'

# simulations_folder_path = f'/home/david/Documents/code/DDM_vXbis_synthetic_data_identical_drift'
# simulations_folder_path = f'/home/david/Documents/code/DDM_vXbis_synthetic_data_identical_drift_0025'
simulations_folder_path = f'/home/david/Documents/code/DDM_vXbis_synthetic_data_identical_drift_0050'

# simulations_folder_path = f'/home/tom/Code/ddm_simulations/simulations_batches_bis/'


## Generation

In [47]:

generate_simulations = False # PARAMS

for i, n_simulations in enumerate(n_simulations_list):
    
    p_cw_init_range = np.linspace(0,1,100)
    
    if not(generate_simulations):

        break

    
    simulations_batch = run_simulations_batch_random_init(args_dict, n_simulations, p_cw_init_range)

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + i+1}.pkl', 'wb') as file:
        pickle.dump(simulations_batch, file)

# HMM fit

## Parameters

In [48]:
n_to_test = np.arange(2,15)
expected_time = np.ceil(len(n_simulations_list)/jl.cpu_count())
print(f'Expected time : {int(expected_time)} t_job')

Expected time : 13 t_job


## Model fit

In [49]:
fit_models = False # PARAMS

if fit_models:

    for i in range(len(n_simulations_list)):

        n_simulations = n_simulations_list[i]

        with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/simulations_batch_{number_of_simulations_perset}_test_{start_index + i+1}.pkl', 'rb') as file:
            synthetic_data = pickle.load(file)

        reformated_training_sequences, reformated_validation_sequences, reformated_training_sequences_lengths, reformated_validation_sequences_lengths = reformat_synthetic_data(synthetic_data)

        fit_output = infer_best_model_score_parallel(reformated_training_sequences, reformated_validation_sequences, 
                                                    reformated_training_sequences_lengths, reformated_validation_sequences_lengths, 
                                                    n_to_test, fix_probability=True, n_features = None, n_fits=50, n_jobs=5)

        save_path = f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index + i+1}.pkl'

        with open(save_path, 'wb') as file:
            pickle.dump(fit_output, file)
    
    

In [110]:
# example_hmm_number = 1

# save_path = f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{example_hmm_number}.pkl'

# with open(save_path, 'rb') as file:
#     fit_output = pickle.load(file)


## Plots

### Plotting initial probability distribution

In [50]:
example_set = 8

with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{example_set}.pkl', 'rb') as file:
    synthetic_data = pickle.load(file)

with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{example_set}.pkl', 'rb') as file:
    fit_output = pickle.load(file)

model = fit_output['models'][6]
test_data = [synth_data['choices'] for synth_data in synthetic_data]
test_proba = [synth_data['p_cw'] for synth_data in synthetic_data]

########################
### Reformating Data ###
########################


emissionmat = model.emissionprob_
startprob = model.startprob_
# # transmat = model.transmat_

# initial_states_arr = np.array(count_initial_state_occurences(model,test_data))

# infered_initial_prob_arr = np.array([])

# for s in initial_states_arr:

#     proba = emissionmat[s,1]
#     infered_initial_prob_arr = np.append(infered_initial_prob_arr, proba)

initial_prob_arr = np.array([])

for p_sequence in test_proba:

    initial_prob_arr = np.append(initial_prob_arr,p_sequence[0])

fig = plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)
ax = plt.subplot(row[0,0])

bins = np.linspace( emissionmat[0,1] - (emissionmat[1,1] - emissionmat[0,1])/2,
                    emissionmat[-1,1] + (emissionmat[1,1] - emissionmat[0,1])/2,
                    len(emissionmat)+1)

binned_initial_prob_arr = np.histogram(initial_prob_arr, bins)[0]/len(initial_prob_arr)

ax.bar(emissionmat[:,1], binned_initial_prob_arr, width=emissionmat[1,1]-emissionmat[0,1], alpha=0.5, label='Simulations initial proba distribution')
ax.bar(emissionmat[:,1], startprob, width=emissionmat[1,1]-emissionmat[0,1], alpha=0.5, label='HMM initial state population')

ax.set_yticks(np.arange(0,1,0.2))
ax.set_ylim((0,0.5))
ax.set_xlabel('Starting probability')
ax.set_ylabel('Fraction')
ax.legend()

### Predict and plot state sequence

In [51]:
example_set = 8

with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{example_set}.pkl', 'rb') as file:
    synthetic_data = pickle.load(file)

with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{example_set}.pkl', 'rb') as file:
    fit_output = pickle.load(file)

model = fit_output['models'][10]
simu_index = 1
choices_sequence = [synth_data['choices'] for synth_data in synthetic_data][simu_index]
rewards_sequence = [synth_data['rewards'] for synth_data in synthetic_data][simu_index]
proba_sequence = [synth_data['p_cw'] for synth_data in synthetic_data][simu_index]

emissionmat = model.emissionprob_
startprob = model.startprob_

predicted_states_sequence = model.predict(np.reshape(choices_sequence,(-1,1)))
predicted_proba_sequence = np.array([emissionmat[s,1] for s in predicted_states_sequence])
colors = ('indigo','violet')
colors_sequence = [colors[r] for r in rewards_sequence]

fig = plt.figure(figsize=(7, 2), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)
ax = plt.subplot(row[0,0])

# ax.scatter(np.arange(len(predicted_proba_sequence)), predicted_proba_sequence, marker='+', c='green')
ax.scatter(np.arange(len(proba_sequence)), proba_sequence, marker='+', c=colors_sequence, alpha=0.7)

ax.plot(np.arange(len(predicted_proba_sequence)), predicted_proba_sequence, marker='+', c='green', alpha=0.5)
ax.plot(np.arange(len(proba_sequence)), proba_sequence, color='k', alpha=0.5)

ax.set_yticks(np.arange(0,1.1,0.2))
# ax.set_ylim((0,0.5))
ax.set_xlabel('Starting probability')
ax.set_ylabel('Fraction')
ax.legend()

/tmp/ipykernel_7730/2346907783.py:38: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend()


### Compute and plot slope per state

In [9]:
def compute_hmm_slope(transmat, emissionmat):

    slope_list = []
    proba_vect = emissionmat[:,1]

    for i in range(len(transmat)):

        slope = np.sum(transmat[i,:] * proba_vect) - proba_vect[i]
        slope_list.append(slope)

    return slope_list

example_number_of_states_index = 3

fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

for example_set in np.arange(len(n_simulations_list))+1:

    with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{example_set}.pkl', 'rb') as file:
        fit_output = pickle.load(file)

    model = fit_output['models'][example_number_of_states_index]

    slopes_list = compute_hmm_slope(model.transmat_, model.emissionprob_)

    # ax1.plot(model.emissionprob_[:,1],slopes_list)
    ax1.plot(model.emissionprob_[1:-1,1],slopes_list[1:-1], alpha=0.5)
    ax1.set_xlabel('State')
    ax1.set_ylabel('Average P_n+1(CW)')

# print(np.mean(slopes_list))

In [71]:
def compute_hmm_up_proba(transmat):

    up_proba_list = []

    for i in range(len(transmat)):

        up_proba = np.sum(transmat[i,i+1:])
        up_proba_list.append(up_proba)

    return up_proba_list

def compute_hmm_down_proba(transmat):

    down_proba_list = []

    for i in range(len(transmat)):

        down_proba = np.sum(transmat[i,:i])
        down_proba_list.append(down_proba)

    return down_proba_list

def compute_hmm_same_proba(transmat):

    same_proba_list = []

    for i in range(len(transmat)):

        same_proba = transmat[i,i]
        same_proba_list.append(same_proba)

    return same_proba_list

example_number_of_states_index = 11

fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(2, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])
ax2 = plt.subplot(row[1,0])

stacked_up_proba_list = []
stacked_down_proba_list = []
stacked_same_proba_list = []
stacked_transmats = []

for example_set in np.arange(len(n_simulations_list))+1:

    with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{example_set}.pkl', 'rb') as file:
        fit_output = pickle.load(file)

    model = fit_output['models'][example_number_of_states_index]

    up_proba_list = compute_hmm_up_proba(model.transmat_)
    down_proba_list = compute_hmm_down_proba(model.transmat_)
    same_proba_list = compute_hmm_same_proba(model.transmat_)

    stacked_up_proba_list.append(up_proba_list)
    stacked_down_proba_list.append(down_proba_list)
    stacked_same_proba_list.append(same_proba_list)
    stacked_transmats.append(model.transmat_)


ax1.plot(model.emissionprob_[1:-1,1], np.mean(stacked_up_proba_list, axis=0)[1:-1], marker='+', label="P_n+1(CW) > P_n(CW)")
ax1.plot(model.emissionprob_[1:-1,1], np.mean(stacked_down_proba_list, axis=0)[1:-1], marker='+', label="P_n+1(CW) < P_n(CW)")
ax1.plot(model.emissionprob_[1:-1,1], np.mean(stacked_same_proba_list, axis=0)[1:-1], marker='+', label="P_n+1(CW) = P_n(CW)")
ax1.axhline(0.5, color='grey',linestyle='--')
ax1.set_xlim([0,1])
ax1.set_ylim([0,1])
ax1.set_title(f"{len(model.emissionprob_)} states")
ax1.legend()
#########

average_transmat = np.mean(stacked_transmats, axis=0)
average_slope = compute_hmm_slope(model.transmat_, model.emissionprob_)

ax2.plot(model.emissionprob_[1:-1,1], average_slope[1:-1], marker='+')
ax2.axhline(0., color='grey',linestyle='--')
ax2.set_xlim([0,1])
ax2.set_xlabel("Probability (given by the state)")
ax2.set_ylabel("Average P_n+1(CW)\ngiven P_n(CW)")

# ax2.set_ylim([0,1])

print(np.mean(stacked_up_proba_list,axis=0))

[0.85997866 0.71179627 0.68896741 0.58528144 0.54596449 0.59057219
 0.53409346 0.54959371 0.39934908 0.36279609 0.34843218 0.17479861
 0.        ]


In [16]:
mean_slopes_list = []
for i in range(len(n_simulations_list)):

    with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index+i+1}.pkl', 'rb') as file:
        temp_fit_output = pickle.load(file)

    model = temp_fit_output['models'][9]
    test_data = [synth_data['choices'] for synth_data in synthetic_data]
    test_proba = [synth_data['p_cw'] for synth_data in synthetic_data]

    slopes_list = compute_hmm_slope(model.transmat_, model.emissionprob_)

    mean_slopes_list.append(np.mean(slopes_list))

fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

ax1.hist(mean_slopes_list,bins=20)

(array([ 2.,  1.,  1.,  2.,  3.,  2.,  4.,  7.,  8., 10., 10.,  6., 12.,
        11.,  6.,  7.,  2.,  3.,  2.,  1.]),
 array([-0.11485662, -0.10176861, -0.08868061, -0.0755926 , -0.0625046 ,
        -0.04941659, -0.03632859, -0.02324059, -0.01015258,  0.00293542,
         0.01602343,  0.02911143,  0.04219944,  0.05528744,  0.06837545,
         0.08146345,  0.09455146,  0.10763946,  0.12072747,  0.13381547,
         0.14690348]),
 <BarContainer object of 20 artists>)

### Plot score vs. number of states

In [10]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(2, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

best_states_nbr_list = []

for i in range(len(n_simulations_list)):

    save_path = f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index + i+1}.pkl'

    with open(save_path, 'rb') as file:
            fit_output = pickle.load(file)

    ax1.plot(n_to_test, fit_output['scores'], alpha=0.2)
    # ax1.scatter(len(fit_output[i][0].transmat_), fit_output[i][1], marker='+', alpha=0.2)

    best_states_nbr_list.append(n_to_test[np.argmax(fit_output['scores'])])

ax1.set_xticks(n_to_test)
ax1.set_xlabel("Number of states")
ax1.set_ylabel("log fit score")

ax2 = plt.subplot(row[1,0])

ax2.hist(best_states_nbr_list, bins=n_to_test, align='left')
ax2.set_xlim([1,n_to_test[-1]+1])
ax2.set_xlabel('Number of states')
ax2.set_ylabel('Number of HMM')


Text(0, 0.5, 'Number of HMM')

### Plot average matrix

In [ ]:
index_nb_of_states = 4

fixed_state_models_list = []

for j in range(len(n_simulations_list)):

    with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index+j+1}.pkl', 'rb') as file:
        fit_output = pickle.load(file)

    transmat_to_add = fit_output['models'][index_nb_of_states].transmat_
    fixed_state_models_list.append(transmat_to_add)

average_transmat = np.mean(fixed_state_models_list, axis=0)
var_transmat = np.std(fixed_state_models_list, axis=0)


fig=plt.figure(figsize=(3.5, 3), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[0].subgridspec(1,3, width_ratios=(1,1,0.1))
ax1 = plt.subplot(row[0,0])
ax2 = plt.subplot(row[0,1])
ax3 = plt.subplot(row[0,2])

# ax1.imshow(average_transmat, vmin=0, vmax=1)

im1 = ax1.imshow(average_transmat, vmin=0, vmax=1)
ax1.set_xticks(np.arange(len(average_transmat)))
ax1.set_yticks(np.arange(len(average_transmat)))
ax1.set_title('Average transition matrix\nover 100 simulations sets', fontsize=5)
ax1.set_xlabel('To state')
ax1.set_ylabel('From state')

im2 = ax2.imshow(var_transmat, vmin=0, vmax=1)
ax2.set_xticks(np.arange(len(average_transmat)))
ax2.set_yticks(np.arange(len(average_transmat)))
ax2.set_title('Standard deviation of transition matrices\nover 100 simulations sets', fontsize=5)

plt.colorbar(im2,cax=ax3)

In [12]:
for i in range(len(n_simulations_list)):

    with open(f'{simulations_folder_path}/n_{number_of_simulations_perset}/model_score_fit_output_{number_of_simulations_perset}_test_{start_index+i+1}.pkl', 'rb') as file:
        temp_fit_output = pickle.load(file)

    model = temp_fit_output['models'][2]

    print(model.monitor_.converged)

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
